# Importiamo il dataset

In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('titanic_train.csv')

# Controlliamo il dataset e creiamo le prime aspettative veloci

In [2]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Controllare valori nulli e valori stringa

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


5 colonne stringa da convertire

Age e Cabin con un po' di valori nulli

Embarked ha pochi valori nulli

## Controllo le statistiche

In [4]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


Min e Max risalta solo Fare con Max alto

Media e 50% risalta Survived, SibSip, Parch, Fare

Media e std risalta SibSip, Parch

## Per evitare leakage prima di fare qualsiasi cosa divido in train e test 

In [12]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Survived'])
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y # serve per avere classi bilanciate tra train e test, se il dataset ha 10% A e 90% B, train e test avranno 10% A e 90% B
)

# Una volta diviso posso procedere a sistemare le cose in ordine: valori nulli, conversione stringhe, rimozione outlier, normalizzazione

## Iniziamo dalla gestione dei nulli

Se i valori nulli sono pochi < 7% rimuoviamo le righe nulle

Se i vlaori nulli sono tanti > 50% rimuoviamo la colonna

Se i valori nulli ci sono ma sono contenuti, < 20/30% cerco un modo per stimare i nulli

In [11]:
# 1. Gestione Embarked: Rimuoviamo le righe con valori nulli in Embarked
# È importante farlo su entrambi i set per mantenere la coerenza
X_train = X_train.dropna(subset=['Embarked'])
y_train = y_train.loc[X_train.index] # Allineiamo il target dopo il drop

X_test = X_test.dropna(subset=['Embarked'])
y_test = y_test.loc[X_test.index]

# 2. Gestione Cabin: Rimuoviamo la colonna intera
# Poiché ha troppi valori mancanti, non è informativa
X_train = X_train.drop(columns=['Cabin'])
X_test = X_test.drop(columns=['Cabin'])

# Verifica veloce
print(f"Righe rimanenti in Train: {len(X_train)}")
print(f"Colonne rimanenti: {X_train.columns.tolist()}")

Righe rimanenti in Train: 710
Colonne rimanenti: ['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Embarked']


## Dobbiamo capire come stimare i valori nulli di age, possiamo fare un vero e proprio modello di ml oppure possiamo fare una matrice di correlazione e vedere se age e' correlato a qualche variabile

In [14]:
# Select only numerical columns for correlation
numerical_df = X_train.select_dtypes(include=['int64', 'float64'])

# Compute correlation matrix
corr_matrix = numerical_df.corr()

# Isolate and sort correlations with respect to 'Age'
age_correlations = corr_matrix['Age'].sort_values(ascending=False)

print("Correlation with Age:")
print(age_correlations)

Correlation with Age:
Age            1.000000
Fare           0.106915
PassengerId    0.036080
Parch         -0.175573
SibSp         -0.312885
Pclass        -0.351089
Name: Age, dtype: float64


## Raggruppa le eta non nulle per Pclass, per ogni valore di Pclass calcola la mediana, e sostituisco le eta nulle con la mediana associata alla loro pclass, calcolo le cose su xtrain  ma le devo applicare anche a xtest

In [21]:
# 1. Calcoliamo la mediana (mapping) basata solo su X_train
# age_median_map è già calcolato in precedenza dal train
age_median_map = X_train.groupby(['Pclass', 'SibSp', 'Parch'])['Age'].median()

# 2. Definiamo la funzione di imputazione che usa il mapping calcolato sul train
def impute_age_advanced(row, median_map, fallback_map):
    if pd.isnull(row['Age']):
        keys = (row['Pclass'], row['SibSp'], row['Parch'])
        if keys in median_map:
            return median_map[keys]
        else:
            # Fallback alla mediana della classe, calcolata precedentemente sul train
            return fallback_map.get(row['Pclass'], 28.0) # 28.0 come backup di sicurezza estremo
    return row['Age']

# 3. Applichiamo la trasformazione
# Per X_train è già fatto, ma per coerenza lo riapplichiamo o verifichiamo:
X_train['Age'] = X_train.apply(lambda row: impute_age_advanced(row, age_median_map, age_by_pclass), axis=1)
X_test['Age'] = X_test.apply(lambda row: impute_age_advanced(row, age_median_map, age_by_pclass), axis=1)

# Verifica finale
print(f"Nulli in X_train: {X_train['Age'].isnull().sum()}")
print(f"Nulli in X_test: {X_test['Age'].isnull().sum()}")

# 4. Stampa formattata dell'intero set (Train + Test)
unique_ages_clean = [float(age) for age in sorted(X_train['Age'].unique())]
import textwrap
print("\nDistribuzione età (Valori unici nel set di training):")
print("-" * 50)
print(textwrap.fill(", ".join([f"{age:.1f}" for age in unique_ages_clean]), width=80))

Nulli in X_train: 0
Nulli in X_test: 0

Distribuzione età (Valori unici nel set di training):
--------------------------------------------------
0.4, 0.7, 0.8, 0.8, 0.9, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0,
11.0, 12.0, 13.0, 14.0, 14.5, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 20.5, 21.0,
22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 28.5, 29.0, 30.0, 30.5, 31.0, 32.0,
32.5, 33.0, 34.0, 34.5, 35.0, 36.0, 36.5, 37.0, 38.0, 39.0, 40.0, 40.5, 41.0,
42.0, 43.0, 44.0, 45.0, 45.5, 46.0, 47.0, 48.0, 49.0, 50.0, 51.0, 52.0, 54.0,
55.0, 55.5, 56.0, 57.0, 58.0, 59.0, 60.0, 61.0, 62.0, 63.0, 64.0, 65.0, 66.0,
70.0, 70.5, 71.0, 74.0, 80.0


# NOTA!!

I modelli avanzati si gestiscono da soli i nulli quindi non dovrete fare queste cose per i modelli di boosting

# ABBIAMO PULITO I DATI, ADESSO CONVERTIAMO LE STRINGHE

## Per prima cosa vediamo le stringhe quanti valori unici hanno

In [25]:
# Selezioniamo le colonne di tipo object/str
string_cols = X_train.select_dtypes(include=['object', 'string']).columns

# Analizziamo il numero di valori univoci per ogni colonna selezionata
unique_counts = X_train[string_cols].nunique()

print("Numero di valori univoci per le colonne stringa:")
print("-" * 40)
print(unique_counts)

Numero di valori univoci per le colonne stringa:
----------------------------------------
Name        712
Sex           2
Ticket      571
Cabin       127
Embarked      3
dtype: int64


Una stringa con 700 valori unici che non hanno rapporti di relazione (maggiore/minore etc...) su 900 serve a qualcosa? No, qiundi elimino la colonna NAME, in altri posso provare conversioni particolari, se fossero nomi di citta potrei sostituire con il market cap e il numero di abitanti prendendo le info da internet

Sex ottimo lo converto in 0 e 1

Ticket... dobbiamo capire meglio cosa contiene la colonna

Cabin possiamo convertirla anche qui capiamo meglio cosa contiene

Embarked lo convertiamo in colonne binarie embarked_A embarked_B embarked_C perche non c'e' una relazione di importanza tra i porti

In [27]:
# Analizziamo 10 valori per capire il pattern
print("--- 10 Esempi di Ticket ---")
print(X_train['Ticket'].head(10))

print("\n--- 10 Esempi di Cabin ---")
# Cabin ha molti NaN, ne prendiamo 10 che siano validi
print(X_train[X_train['Cabin'].notnull()]['Cabin'].head(10))

--- 10 Esempi di Ticket ---
692          1601
481        239854
527      PC 17483
855        392091
801    C.A. 31921
652          8475
509          1601
557      PC 17757
828        367228
18         345763
Name: Ticket, dtype: str

--- 10 Esempi di Cabin ---
527      C95
647      A26
412      C78
218      D15
835      E49
430      C52
449     C104
96        A5
715    F G73
434      E44
Name: Cabin, dtype: str


In ticked si potrebbe dividere la parte stringa dalla aprte numerica e creare due colonne partendo da una, i valori numerici sono gestibili dal modello i valori stringa vanno analizzati alla ricerca di pattern. Andrebbero cercati i significati della stringa del ticked e andrebbe vista la rilevanza statistica del dato (se troppo pochi e' inutile)

Di cabin terrei soltanto la lettera che rappresenza la zona, da valutare la rilevanza statistica del numero tramite correlazioni


## Per convertire le stringhe abbiamo due strade: ONE HOT crea una colonna binaria per valore, Label crea una colonna singola conviene se abbiamno relazioni di importanza tra le stringhe, come piccolo, medio, grande oppure se abbiamo veramente troppe stringhe 200+ per cui andare a creare una colonna per ogni stringa aumenta troppo la dimensionalita del dataset. Se trasformo una colonna in 200 aumento troppo la grandezza del dataset

Rimuovo name, cabin e ticket e converto sex e embarked in one hot (una colonna per ogni valore)

In [28]:
# 1. Rimozione colonne non necessarie
cols_to_drop = ['Name', 'Ticket', 'Cabin']
X_train = X_train.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

# 2. Conversione Sex (Binaria: female=0, male=1)
# Utilizziamo .map per coerenza su entrambi i set
sex_map = {'female': 0, 'male': 1}
X_train['Sex'] = X_train['Sex'].map(sex_map)
X_test['Sex'] = X_test['Sex'].map(sex_map)

# 3. Conversione Embarked (One-Hot Encoding)
# Usiamo get_dummies per creare le colonne separate
X_train = pd.get_dummies(X_train, columns=['Embarked'], prefix='Embarked')
X_test = pd.get_dummies(X_test, columns=['Embarked'], prefix='Embarked')

# Verifica finale delle colonne e dei tipi
print("Colonne risultanti:")
print(X_train.columns.tolist())
print("\nPrime 5 righe (Train):")
print(X_train.head())
print(f"\nTipi di dati:\n{X_train.dtypes}")

Colonne risultanti:
['PassengerId', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked_C', 'Embarked_Q', 'Embarked_S']

Prime 5 righe (Train):
     PassengerId  Pclass  Sex   Age  SibSp  Parch      Fare  Embarked_C  \
692          693       3    1  24.0      0      0   56.4958       False   
481          482       2    1  29.0      0      0    0.0000       False   
527          528       1    1  38.0      0      0  221.7792       False   
855          856       3    0  18.0      0      1    9.3500       False   
801          802       2    0  31.0      1      1   26.2500       False   

     Embarked_Q  Embarked_S  
692       False        True  
481       False        True  
527       False        True  
855       False        True  
801       False        True  

Tipi di dati:
PassengerId      int64
Pclass           int64
Sex              int64
Age            float64
SibSp            int64
Parch            int64
Fare           float64
Embarked_C        bool
Embarked_Q        